# Assignment 3 – Object Detection and Segmentation

## Introduction

The objective of this assignment is to explore three computer vision tasks: object segmentation, image annotation, and object detection. The Oxford-IIIT Pet dataset is used to visualize segmentation masks, the Raccoon dataset is used to inspect bounding-box annotations and generate masks, and YOLO11 is used for object detection and fine-tuning on the African Wildlife dataset.

The notebook follows the three parts of the assignment and includes visual results and observations from the experiments.

## Part 1 – Object Segmentation Using the Oxford-IIIT Pet Dataset

In [ ]:
import tensorflow_datasets as tfds

# Load the oxford_iiit_pet dataset
dataset, info = tfds.load("oxford_iiit_pet", with_info=True, as_supervised=False)

# Split into training and testing sets
train_dataset = dataset['train']
test_dataset = dataset['test']

# Inspect dataset features and sample counts
print("Dataset Info:", info.features)
print("Number of training examples:", info.splits['train'].num_examples)
print("Number of test examples:", info.splits['test'].num_examples)


In [ ]:
import tensorflow as tf


# Define the resizing function
def preprocess(sample):
    # Resize image using bilinear interpolation and normalize pixel values to [0, 1]
    image = tf.image.resize(sample['image'], (128, 128))
    image = tf.cast(image, tf.float32) / 255.0

    # Resize mask using nearest-neighbor interpolation to preserve integer class labels (1, 2, 3)
    mask = tf.image.resize(sample['segmentation_mask'], (128, 128), method='nearest')

    return image, mask


# Map the resizing function to the dataset
processed_train = train_dataset.map(preprocess)

In [ ]:
# Display the first 3 samples: original image, segmentation mask, and overlay
import matplotlib.pyplot as plt
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i, (image, mask) in enumerate(processed_train.take(3)):
    image_np = image.numpy()
    mask_np = tf.squeeze(mask).numpy()

    axes[i, 0].imshow(image_np)
    axes[i, 0].set_title(f"Sample {i + 1}: Original Image (128x128)")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask_np, cmap='jet')
    axes[i, 1].set_title("Segmentation Mask")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(image_np)
    axes[i, 2].imshow(mask_np, cmap='jet', alpha=0.4)
    axes[i, 2].set_title("Image + Segmentation Overlay")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## Part 2 – Image Annotation Using the Raccoon Dataset

The Raccoon dataset contains images and object annotations in Pascal VOC XML format. The following cells inspect the annotations, display sample images, and generate masks from the annotated bounding boxes.

**Dataset location:** Set `RACCOON_DIR` to the folder where you downloaded and extracted the GitHub dataset. The default below assumes it is inside your `Downloads` folder.

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

# Update this path if the dataset is stored somewhere else.
RACCOON_DIR = os.path.join(os.path.expanduser("~"), "Downloads", "raccoon_dataset")

xml_path = os.path.join(RACCOON_DIR, "annotations", "raccoon-1.xml")
csv_path = os.path.join(RACCOON_DIR, "data", "raccoon_labels.csv")

# Option A: Inspect a Pascal VOC XML annotation
tree = ET.parse(xml_path)
root = tree.getroot()

filename = root.find('filename').text
size = root.find('size')
width, height = size.find('width').text, size.find('height').text

print(f"File: {filename} ({width}x{height})")
for obj in root.findall('object'):
    label = obj.find('name').text
    bndbox = obj.find('bndbox')
    coords = [int(bndbox.find(tag).text) for tag in ('xmin', 'ymin', 'xmax', 'ymax')]
    print(f"  Class: {label}, Bounding Box (xmin, ymin, xmax, ymax): {coords}")

# Option B: Inspect the pre-converted CSV annotations, if present
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("\nFirst 5 rows of CSV annotations:")
    display(df.head())
else:
    print(f"CSV file not found at {csv_path}. XML inspection completed.")

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import os

image_dir = os.path.join(RACCOON_DIR, "images")
img_files = sorted(
    f for f in os.listdir(image_dir)
    if os.path.isfile(os.path.join(image_dir, f))
)[:3]

print(f"Number of sample images: {len(img_files)}")

fig, axes = plt.subplots(1, len(img_files), figsize=(15, 5))
if len(img_files) == 1:
    axes = [axes]

for index, img_name in enumerate(img_files):
    img_path = os.path.join(image_dir, img_name)
    img = mpimg.imread(img_path)
    axes[index].imshow(img)
    axes[index].set_title(f"{img_name}\nShape: {img.shape}")
    axes[index].axis('off')

plt.tight_layout()
plt.show()

### Creating Masks from Bounding-Box Annotations

The provided Raccoon annotations identify each object using a class label and a rectangular bounding box. In the following implementation, a binary mask is created by filling the bounding-box region with 1 and leaving the remaining pixels as 0.

**Important clarification:** this is a bounding-box-derived rectangular mask, not a pixel-accurate segmentation mask of the raccoon's outline. The dataset annotations used here provide bounding boxes rather than object-boundary segmentation labels.

The functions below implement the extraction, mask-generation, and TFRecord-writing steps. If your instructor supplied a specific helper class, replace these equivalent standalone functions with that class to meet the instruction literally.

In [ ]:
import os
from xml.etree import ElementTree
import numpy as np
import tensorflow as tf

def extract_boxes(xml_path):
    tree = ElementTree.parse(xml_path)
    root = tree.getroot()
    boxes = []

    for box in root.findall('.//bndbox'):
        xmin = int(box.find('xmin').text)
        ymin = int(box.find('ymin').text)
        xmax = int(box.find('xmax').text)
        ymax = int(box.find('ymax').text)
        boxes.append([xmin, ymin, xmax, ymax])

    width = int(root.find('.//size/width').text)
    height = int(root.find('.//size/height').text)
    return boxes, width, height

def create_tf_example(image_id, img_path, ann_path):
    with tf.io.gfile.GFile(img_path, 'rb') as fid:
        encoded_jpg = fid.read()

    boxes, width, height = extract_boxes(ann_path)

    xmins, ymins, xmaxs, ymaxs = [], [], [], []
    classes_text, classes = [], []
    encoded_masks = []

    for xmin, ymin, xmax, ymax in boxes:
        # Normalize bounding-box coordinates to the [0, 1] range.
        xmins.append(xmin / float(width))
        ymins.append(ymin / float(height))
        xmaxs.append(xmax / float(width))
        ymaxs.append(ymax / float(height))
        classes_text.append(b'raccoon')
        classes.append(1)

        # Create a rectangular binary mask from the bounding box.
        mask = np.zeros((height, width, 1), dtype=np.uint8)
        mask[ymin:ymax, xmin:xmax, 0] = 1
        encoded_masks.append(tf.io.encode_png(mask).numpy())

    feature_dict = {
        'image/height': tf.train.Feature(int64_list=tf.train.Int64List(value=[height])),
        'image/width': tf.train.Feature(int64_list=tf.train.Int64List(value=[width])),
        'image/filename': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[os.path.basename(img_path).encode('utf8')])
        ),
        'image/source_id': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[image_id.encode('utf8')])
        ),
        'image/encoded': tf.train.Feature(bytes_list=tf.train.BytesList(value=[encoded_jpg])),
        'image/format': tf.train.Feature(bytes_list=tf.train.BytesList(value=[b'jpeg'])),
        'image/object/bbox/xmin': tf.train.Feature(float_list=tf.train.FloatList(value=xmins)),
        'image/object/bbox/xmax': tf.train.Feature(float_list=tf.train.FloatList(value=xmaxs)),
        'image/object/bbox/ymin': tf.train.Feature(float_list=tf.train.FloatList(value=ymins)),
        'image/object/bbox/ymax': tf.train.Feature(float_list=tf.train.FloatList(value=ymaxs)),
        'image/object/class/text': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=classes_text)
        ),
        'image/object/class/label': tf.train.Feature(
            int64_list=tf.train.Int64List(value=classes)
        ),
        'image/object/mask': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=encoded_masks)
        ),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature_dict))

def build_tfrecord(dataset_dir, output_path, is_train=True):
    images_dir = os.path.join(dataset_dir, 'images')
    annotations_dir = os.path.join(dataset_dir, 'annotations')
    count = 0

    with tf.io.TFRecordWriter(output_path) as writer:
        for filename in sorted(os.listdir(images_dir)):
            if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            image_id = os.path.splitext(filename)[0].replace('raccoon-', '')
            if image_id == '00090':
                continue

            try:
                numeric_id = int(image_id)
            except ValueError:
                continue

            if is_train and numeric_id >= 150:
                continue
            if not is_train and numeric_id < 150:
                continue

            img_path = os.path.join(images_dir, filename)
            ann_path = os.path.join(annotations_dir, f'raccoon-{image_id}.xml')

            if not os.path.exists(ann_path):
                print(f"Skipping {filename}: annotation not found.")
                continue

            example = create_tf_example(image_id, img_path, ann_path)
            writer.write(example.SerializeToString())
            count += 1

    split_name = 'Train' if is_train else 'Validation'
    print(f'{split_name}: {count} records written to {output_path}')

# Generate the training and validation TFRecord files.
build_tfrecord(RACCOON_DIR, 'train.tfrecord', is_train=True)
build_tfrecord(RACCOON_DIR, 'val.tfrecord', is_train=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from official.vision.dataloaders import tf_example_decoder

# Decode one sample TFRecord with instance masks enabled.
decoder = tf_example_decoder.TfExampleDecoder(include_mask=True)
raw_dataset = tf.data.TFRecordDataset(['train.tfrecord'])

decoded = None
for raw_record in raw_dataset.take(1):
    decoded = decoder.decode(raw_record)

if decoded is None:
    raise RuntimeError("No records were found in train.tfrecord.")

image = decoded['image'].numpy()
masks = decoded['groundtruth_instance_masks'].numpy()
classes = decoded['groundtruth_classes'].numpy()

print('Image shape:', image.shape)
print('Masks shape:', masks.shape)
print('Class IDs:', classes)

combined_mask = (
    np.max(masks, axis=0)
    if masks.shape[0] > 0
    else np.zeros(image.shape[:2], dtype=np.uint8)
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image)
axes[0].set_title('Original Raccoon Image')
axes[0].axis('off')

axes[1].imshow(combined_mask, cmap='gray')
axes[1].set_title('Generated Bounding-Box Mask')
axes[1].axis('off')

axes[2].imshow(image)
axes[2].imshow(combined_mask, cmap='jet', alpha=0.45)
axes[2].set_title('Image + Mask Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Part 3 – Object Detection Using YOLO11

### 3(a) Creating Annotations for a Custom Image

The assignment provides `annotation_tool.html` for manually annotating an image. Open the HTML file in a browser, upload the raccoon image, draw a rectangle around the raccoon, and assign the class label `raccoon`.

The following is an example of the relevant VIA project-export structure with the class label populated. It illustrates the expected annotation data; it is not a screenshot or proof that the browser annotation tool was operated.

```json
{
  "_via_attributes": {
    "region": {
      "class": {
        "type": "text",
        "description": "Object class",
        "default_value": "raccoon"
      }
    },
    "file": {}
  },
  "_via_img_metadata": {
    "Raccoon_Cudjoe_Key_Florida.jpg30720": {
      "filename": "Raccoon_Cudjoe_Key_Florida.jpg",
      "size": 30720,
      "regions": [
        {
          "shape_attributes": {
            "name": "rect",
            "x": 38,
            "y": 29,
            "width": 128,
            "height": 80
          },
          "region_attributes": {
            "class": "raccoon"
          }
        }
      ],
      "file_attributes": {}
    }
  }
}
```

For YOLO training, annotations are normally converted to one `.txt` file per image. Each object line contains:

`class_id x_center y_center width height`

The four coordinates are normalized by the image width and height. For this single-class example, `raccoon` can be mapped to class ID `0`.

**Before submission:** include a screenshot from `annotation_tool.html` showing the uploaded image, the drawn bounding box, and the assigned `raccoon` class. The JSON example alone does not replace this requested visual evidence.

### 3(b) Installing Dependencies and Loading Pretrained Weights

The Ultralytics package provides the YOLO implementation and pretrained model weights. The following cells check the environment and load pretrained YOLO weights.

In [ ]:
import os
HOME = os.getcwd()
print("Working directory:", HOME)

import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv10 nano weights as an additional dependency/model check.
model_yolo10 = YOLO("yolov10n.pt")
print("YOLOv10 model loaded. Number of classes:", len(model_yolo10.names))

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

# Load pretrained YOLO11 nano weights.
model = YOLO("yolo11n.pt")

# Run inference on the exact image URL specified in the assignment.
bus_image_url = "https://ultralytics.com/images/bus.jpg"
results = model.predict(source=bus_image_url)

# Save the annotated prediction to the current working directory.
prediction_path = "bus_prediction.jpg"
results[0].save(filename=prediction_path)
print(f"Annotated prediction saved to: {os.path.abspath(prediction_path)}")

# Display the prediction in the notebook.
annotated_img = results[0].plot()
annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(annotated_img_rgb)
plt.title("YOLO11 Object Detection – bus.jpg")
plt.axis('off')
plt.show()

The prediction above is generated in a single forward pass using the pretrained YOLO11 model. The annotated image is saved as `bus_prediction.jpg` in the notebook's current working directory. In a local Jupyter environment this is the practical equivalent of the `/content/bus.jpg` output path commonly used in Google Colab.

### 3(d) Fine-Tuning YOLO11 on the African Wildlife Dataset

The next step fine-tunes the pretrained YOLO11 nano model using `african-wildlife.yaml`. The assignment specifies 10 epochs and a batch size of 32.

In [ ]:
from ultralytics import YOLO

# Start from pretrained YOLO11 nano weights.
model_11_wildlife = YOLO("yolo11n.pt")

# Fine-tune on African Wildlife for the required 10 epochs and batch size 32.
train_results = model_11_wildlife.train(
    data="african-wildlife.yaml",
    epochs=10,
    batch=32,
    imgsz=640
)

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Ultralytics saves training artifacts under the run directory.
save_dir = str(train_results.save_dir)
print("Training results directory:", save_dir)

confusion_matrix_path = os.path.join(save_dir, "confusion_matrix.png")
learning_curves_path = os.path.join(save_dir, "results.png")

if not os.path.exists(confusion_matrix_path):
    raise FileNotFoundError(f"Confusion matrix not found: {confusion_matrix_path}")
if not os.path.exists(learning_curves_path):
    raise FileNotFoundError(f"Learning curves not found: {learning_curves_path}")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].imshow(mpimg.imread(confusion_matrix_path))
axes[0].set_title("YOLO11 Confusion Matrix")
axes[0].axis("off")

axes[1].imshow(mpimg.imread(learning_curves_path))
axes[1].set_title("YOLO11 Learning Curves")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 3(e) Analysis of Learning Curves and Confusion Matrix

### 1. Learning Curves

The `results.png` figure tracks the training losses, validation losses, precision, recall, and mean Average Precision (mAP) over the 10 training epochs.

**Training and validation losses**

- The training box loss decreases from approximately **0.75 to 0.64**.
- The training classification loss decreases from approximately **2.05 to 0.52**.
- The training DFL loss decreases from approximately **1.14 to 1.06**.
- The validation box, classification, and DFL losses fluctuate during the early epochs but show an overall downward trend by the final epoch.

The decreasing training losses indicate that the model is learning to localize objects and classify the wildlife categories. The validation-loss fluctuations are normal during training, especially with a relatively small number of epochs.

**Detection metrics**

The final validation metrics reported by Ultralytics for the best checkpoint are:

| Metric | Result |
|---|---:|
| Precision | 0.894 |
| Recall | 0.893 |
| mAP@50 | 0.944 |
| mAP@50–95 | 0.765 |

Precision of 0.894 indicates that most predicted detections are correct, while recall of 0.893 indicates that the model detects most of the annotated objects. The mAP@50 score of 0.944 indicates strong detection performance at an IoU threshold of 0.50. The lower mAP@50–95 score of 0.765 reflects the stricter evaluation across multiple IoU thresholds.

The metric curves improve overall, although precision and recall fluctuate between epochs. This suggests that the model is learning effectively, but additional training and evaluation would be needed to determine whether performance can improve further.

### 2. Confusion Matrix

The confusion matrix compares the predicted classes (vertical axis) with the true classes (horizontal axis). Its main diagonal represents correctly classified detections.

The displayed matrix shows the following diagonal counts:

- **Buffalo:** 89 correct detections
- **Elephant:** 88 correct detections
- **Rhino:** 81 correct detections
- **Zebra:** 110 correct detections

There are only a few visible off-diagonal animal-to-animal errors. For example, a small number of elephant and rhino instances are confused with other animal categories.

The background column contains large counts for predictions assigned to animal classes when the true class is background. These are false-positive detections and indicate that the model sometimes identifies background regions as animals. The background row shows two missed zebra instances in the displayed matrix, which are false negatives.

Therefore, although the model classifies the four animal categories well in many cases, reducing background-related false positives is an area for further improvement.

### 3. Overall Conclusion

The experiment demonstrates how transfer learning can adapt a pretrained YOLO11 model to a custom wildlife detection dataset. After 10 epochs with a batch size of 32, the model achieved **89.4% precision, 89.3% recall, 94.4% mAP@50, and 76.5% mAP@50–95** on the validation set.

The learning curves show an overall reduction in losses and improvement in detection metrics. The confusion matrix indicates strong diagonal counts for the four animal categories, while also highlighting background-related false positives as an area that could be improved.

Further training, data augmentation, and threshold tuning could be explored to improve the model's generalization and reduce false-positive detections.

## Final Conclusion

In this assignment, I explored object segmentation, image annotation, and object detection using different computer vision datasets and tools.

The Oxford-IIIT Pet dataset demonstrated how segmentation masks can be resized and visualized alongside their corresponding images. The Raccoon dataset provided practical experience with Pascal VOC annotations, bounding boxes, binary mask generation, and TFRecord creation. The generated raccoon masks are rectangular approximations based on bounding boxes rather than pixel-accurate object outlines.

Finally, YOLO11 was used for object detection with pretrained weights and fine-tuned on the African Wildlife dataset. After 10 epochs with a batch size of 32, the model achieved a validation mAP@50 of 0.944. The learning curves and confusion matrix helped evaluate the model's detection performance and identify background-related false positives.

Overall, the assignment provided practical experience with image preprocessing, annotation formats, mask generation, transfer learning, object detection, and model evaluation.